# LegalQA Task 2 — Google Colab A100 Production Training Pipeline (Notion DSC 2026)
Production QLoRA generator training, full validation, and Hugging Face artifact release on NVIDIA A100.
- **Prerequisite**: Kaggle Dual-T4 Smoke Gate must have reached  status for the frozen tuple.
- **Configuration**:  (BF16 native throughput, larger batch size, full training epochs).
- **Outputs**: Full Run Bundle exported and uploaded to Hugging Face repository.

In [ ]:
# Cell 1: Hardware & Environment Verification
import os, sys, subprocess, torch
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".05"
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"

print("=== Hardware Verification ===")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Colab production training.")

gpu_name = torch.cuda.get_device_name(0)
print(f"Detected GPU: {gpu_name}")
is_a100 = "A100" in gpu_name
if not is_a100:
    print(f"Notice: Optimal profile target is NVIDIA A100, currently running on {gpu_name}.")

try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception:
    pass


In [ ]:
# Cell 2: Git Repository Sync & Environment Setup
import os, sys, subprocess

TARGET_GIT_SHA = "main"
code_root = "/content/LegalQA" if os.path.exists("/content") else os.path.abspath(".")

if not os.path.exists(code_root) and not os.path.exists("src/task2"):
    os.system(f"git clone https://github.com/silent9669/LegalQA.git {code_root}")
elif os.path.isdir(code_root) and os.path.exists(os.path.join(code_root, ".git")):
    os.system(f"cd {code_root} && git fetch origin && git reset --hard origin/main")

if TARGET_GIT_SHA != "main":
    os.system(f"cd {code_root} && git checkout {TARGET_GIT_SHA}")

# Evict stale namespace caches
for mod in list(sys.modules.keys()):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

if code_root in sys.path:
    sys.path.remove(code_root)
sys.path.insert(0, code_root)
os.chdir(code_root)

# Install user-space dependencies with exact constraints-gpu.txt
print("Bootstrapping user-space dependencies from constraints-gpu.txt...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-c", "constraints-gpu.txt",
    "transformers", "peft", "accelerate", "datasets", "trl", "liger-kernel", "bitsandbytes", "kaggle", "kagglehub"
], check=True)

# Load environment credentials (.env, Colab secrets, or env vars)
from src.common.env_loader import load_environment
env_status = load_environment()
print(f"Environment Loaded: {env_status.get('loaded_from_file') or 'default/secrets'}")
print(f"Hugging Face Auth: {'CONFIGURED (' + env_status['hf_token_masked'] + ')' if env_status['hf_token_configured'] else 'NOT CONFIGURED'}")
print(f"Kaggle Auth: {'CONFIGURED (' + env_status['kaggle_user'] + ')' if env_status['kaggle_configured'] else 'NOT CONFIGURED'}")

if not env_status['hf_token_configured']:
    print("[!] Notice: HF_TOKEN is not configured. In Colab, add 'HF_TOKEN' to Secrets (key icon) or upload a .env file.")
if not env_status['kaggle_configured']:
    print("[!] Notice: KAGGLE_KEY is not configured. In Colab, add 'KAGGLE_KEY' and 'KAGGLE_USERNAME' to Secrets or upload a .env file.")

print(f"Repository synchronized at: {code_root}")

In [ ]:
# Cell 3: Dataset Mount & Schema Validation
import sys, os
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

from src.task2.dataset.validator import validate_dataset

DATA_DIR = "/content/data/legalqa-task2-clean-data"
if not os.path.exists(DATA_DIR):
    if os.path.exists("/kaggle/input/legalqa-task2-clean-data"):
        DATA_DIR = "/kaggle/input/legalqa-task2-clean-data"
    elif os.path.exists("kaggle_dataset/legal_chunks.parquet"):
        DATA_DIR = os.path.abspath("kaggle_dataset")
    else:
        try:
            import kagglehub
            DATA_DIR = kagglehub.dataset_download("phucdangg/legalqa-task2-clean-data/versions/1")
        except Exception as e:
            print(f"kagglehub download notice: {e}, falling back to kaggle CLI...")
            os.system("kaggle datasets download -d phucdangg/legalqa-task2-clean-data --unzip -p /content/data/legalqa-task2-clean-data")
            DATA_DIR = "/content/data/legalqa-task2-clean-data"

print(f"Active Dataset Directory: {DATA_DIR}")
val_report = validate_dataset(data_dir=DATA_DIR, schema_path="configs/dataset_schema.yaml")
print(f"Dataset Manifest Status: {val_report.get('status')} (Verified: {val_report.get('manifest_verified')})")
if val_report.get('status') != "PASS":
    raise RuntimeError(f"Dataset validation failed: {val_report.get('errors')}")


In [ ]:
# Cell 4: Smoke Pass Gate Verification (strict GateReport chain first, legacy stub last)
import sys, os, json
from pathlib import Path
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

from src.task2.provenance.freeze_tuple import verify_smoke_pass

# Prefer the strict GateReport chain in artifacts/gates (promotable evidence).
# The root kaggle_smoke_report.json is a legacy stub and is NOT sufficient for Lead approval.
SMOKE_REPORT_PATH = None
k_reports = list((Path(code_root) / "artifacts" / "gates").glob("*/kaggle_t4x2_report.json"))
if k_reports:
    SMOKE_REPORT_PATH = str(sorted(k_reports, key=lambda p: p.stat().st_mtime)[-1])
elif os.path.exists("kaggle_smoke_report.json"):
    SMOKE_REPORT_PATH = "kaggle_smoke_report.json"
elif os.path.exists("/content/kaggle_smoke_report.json"):
    SMOKE_REPORT_PATH = "/content/kaggle_smoke_report.json"

if SMOKE_REPORT_PATH is None:
    print("Notice: Proceeding with explicit execution (kaggle_smoke_report.json not attached locally).")
else:
    with open(SMOKE_REPORT_PATH, "r", encoding="utf-8") as _f:
        _data = json.load(_f)
    if "candidate_id" in _data and "stage" in _data:
        from src.task2.provenance.candidate import CandidateManifest
        from src.task2.provenance.gate_report import verify_gate_report
        _cid = _data.get("candidate_id")
        _cand_p = Path(code_root) / "artifacts" / "candidates" / str(_cid) / "candidate_manifest.json"
        if _cand_p.is_file():
            _cand = CandidateManifest.load_json(_cand_p)
            verify_gate_report(SMOKE_REPORT_PATH, _cand, expected_stage="kaggle_t4x2")
            print(f"Verified: strict Kaggle Dual-T4 GateReport PASS for candidate {_cid} from {SMOKE_REPORT_PATH}.")
        else:
            if not verify_smoke_pass(SMOKE_REPORT_PATH):
                raise RuntimeError(f"Kaggle smoke gate report at {SMOKE_REPORT_PATH} did NOT report PASS. Cannot proceed with A100 training.")
            print(f"Verified (weak): smoke PASS from {SMOKE_REPORT_PATH}, but candidate manifest {_cid} not found locally — launch_colab_training.py will enforce the strict chain before A100.")
    else:
        if not verify_smoke_pass(SMOKE_REPORT_PATH):
            raise RuntimeError(f"Kaggle smoke gate report at {SMOKE_REPORT_PATH} did NOT report PASS. Cannot proceed with A100 training.")
        print(f"WARNING: {SMOKE_REPORT_PATH} is a legacy stub (no candidate_id/stage) — NOT sufficient for Lead approval. A100 promotion requires artifacts/gates/<candidate_id>/kaggle_t4x2_report.json verified via launch_colab_training.py.")

# Dense encoder pin for the cold-rebuild gate (cell 5). The checked-in dense
# matrix is quarantined (see artifacts/labs/DENSE_INDEX_QUARANTINE.md), so the
# A100 run must verify or rebuild it against this immutable revision.
DENSE_REVISION = None
try:
    if "_cand" in dir() and _cand is not None:
        DENSE_REVISION = _cand.models.dense.revision
        print(f"Dense encoder pin: {_cand.models.dense.id} @ {DENSE_REVISION}")
    else:
        print("Notice: no candidate manifest in cell 4; cell 5 cannot pin the dense rebuild.")
except Exception as _e:
    print(f"Notice: dense pin unavailable ({_e}).")


In [ ]:
# Cell 5: Execute Colab Production Training (authoritative runtime profile only)
import os, sys, subprocess
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

# Authoritative two-layer config: algorithm.yaml (score-affecting) + runtime/colab_a100.yaml (hardware).
# Legacy flat configs (configs/colab_train_a100.yaml, configs/kaggle_smoke_t4.yaml) are frozen for
# backward compatibility only and MUST NOT be used for production training (lora_r/lr/batch diverge).
config_file = "configs/task2/runtime/colab_a100.yaml" if is_a100 else "configs/task2/runtime/colab_t4.yaml"
if not os.path.exists(config_file):
    raise RuntimeError(f"Authoritative runtime profile missing: {config_file}. Checkout the exact approved commit before training.")

print(f"Hardware-Aligned Profile Target for {gpu_name}: {config_file}")
if "DENSE_REVISION" not in dir() or not DENSE_REVISION:
    raise RuntimeError("Dense encoder pin missing: run cell 4 with the candidate manifest present. Refusing unpinned rebuild.")
cmd = [
    sys.executable,
    "scripts/run_pipeline.py",
    "--config", config_file,
    "--data-dir", DATA_DIR,
    "--output-dir", "/content/runs/current",
    "--allow-single-gpu",
    "--ensure-dense-index",
    "--dense-revision", DENSE_REVISION,
]
print("Executing:", " ".join(cmd))

# Stream process stdout/stderr directly into notebook output
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Pipeline execution returned non-zero exit code: {proc.returncode}")
print("Colab production training finished successfully.")


In [ ]:
# Cell 6: Run Bundle Packaging & Evidence Verification
import glob
print("=== Generated Run Artifacts ===")
for p in sorted(glob.glob("/content/runs/current/**", recursive=True)):
    if os.path.isfile(p):
        print(f" - {p} ({os.path.getsize(p)/1024:.1f} KB)")
print("Run Bundle successfully generated for Hugging Face upload.")


In [ ]:
# Cell 7: Upload Model, Adapter, Checkpoints & Logs to Hugging Face
# Single canonical target (see src/task2/hf_uploader.py::DEFAULT_HF_REPO):
#   dangphuc2109/legalqa-qwen2.5-3b-adapter (public) under runs/<run_id>/
#   Policy is public; the only secret is HF_TOKEN itself (never committed).
import os, sys
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

from src.task2.hf_uploader import DEFAULT_HF_REPO, upload_directory_to_hf

repo_id = DEFAULT_HF_REPO
private = False

print(f"Releasing artifacts to Hugging Face: {repo_id}...")
upload_res = upload_directory_to_hf(
    repo_id=repo_id,
    folder_path="/content/runs/current",
    private=private,
    commit_message=f"feat(colab): release {gpu_name} trained adapter, logs, and checkpoints",
)
print(f"Hugging Face Upload Status: {upload_res['status']}")
print(f"View Model Repository at: {upload_res['repo_url']}")
